In [ ]:
%load_ext autoreload
%autoreload 2 


In [1]:
import wandb
import torch.nn as nn
import torch
import numpy as np
import hydra
import einops
import os
import sys
from pathlib import Path
from dynamics_modeling.eval import batch_eval_loop
from torch.utils.data import DataLoader


os.environ["WANDB_DIR"] = '/pscratch/sd/g/gzhao27/INR/coral/wandb'
os.environ["RESULTS_DIR"] = ''

# sys.path.append(str(Path(__file__).parents[1]))
sys.path.append('/pscratch/sd/g/gzhao27/INR/coral')
sys.path.append('/pscratch/sd/g/gzhao27/INR/SOMA')
from coral.utils.plot import show
from coral.utils.models.scheduling import ode_scheduling
from coral.utils.models.load_inr import create_inr_instance, load_inr_model
from coral.utils.models.get_inr_reconstructions import get_reconstructions
from coralsoma.load_modulations import load_graph_modulations, graph_ode_inr_predict
from coral.utils.data.load_data import get_dynamics_data, set_seed
from coral.utils.data.dynamics_dataset import (KEY_TO_INDEX, TemporalDatasetWithCode)
from coral.mlp import Derivative
from torchdiffeq import odeint
from omegaconf import DictConfig, OmegaConf
from utils.data.unstructure_dataset import GraphNavierStokes, collate_graph_inr, GraphSomaDataset

# with hydra.initialize(config_path="../config/"):
#     # Compose the configuration (equivalent to loading ode.yaml)
#     cfg = hydra.compose(config_name="ode.yaml")

2024-10-14 08:02:56.569181: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-14 08:02:56.582954: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-14 08:02:56.587111: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-14 08:02:56.600140: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-14 08:02:57.854881: W tensorflow/compiler/tf2

In [3]:
# NS task

input_dim = 1
output_dim = 2

# inr
inr_save_name = "NS_test"
inr_save_dir = "/pscratch/sd/g/gzhao27/INR/SOMA/results"
ode_save_name = "NS_test_ode"

ode_save_path = os.path.join(inr_save_dir, ode_save_name+'.pt')
ode_results = torch.load(ode_save_path)

cfg = ode_results['cfg']



/tmp/ipykernel_1247122/2765918221.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ode_results = torch.load(ode_save_path)


In [4]:


torch.set_default_dtype(torch.float32)
data_dir = cfg.data.dir
dataset_name = cfg.data.dataset_name
ntrain = cfg.data.ntrain
ntest = cfg.data.ntest
data_to_encode = cfg.data.data_to_encode
space_factor = cfg.data.space_factor
time_factor = cfg.data.time_factor
seed = cfg.data.seed
same_grid = cfg.data.same_grid
seq_inter_len = cfg.data.seq_inter_len
seq_extra_len = cfg.data.seq_extra_len

missing_rate = cfg.data.missing_rate

inner_steps = cfg.inr.inner_steps

# dynamics
model_type = cfg.dynamics.model_type
hidden = cfg.dynamics.width
depth = cfg.dynamics.depth
epsilon = cfg.dynamics.teacher_forcing_init
epsilon_t = cfg.dynamics.teacher_forcing_decay
epsilon_freq = cfg.dynamics.teacher_forcing_update



In [5]:
input_dim = 2
output_dim = 1


if inr_save_name is not None:
    multichannel = False
    tmp = torch.load(os.path.join(inr_save_dir, inr_save_name+'.pt'))
    latent_dim = tmp["cfg"].inr.latent_dim
    # print('fix sub_tr to space_factor')
    space_factor = tmp["cfg"].data.space_factor
    seed = tmp["cfg"].data.seed
    # missing_rate = tmp["cfg"].data.missing_rate
    
    inr, alpha = load_inr_model(
        inr_save_dir,
        inr_save_name,
        data_to_encode,
        input_dim=input_dim,
        output_dim=output_dim,
    )

/tmp/ipykernel_1247122/2577899674.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tmp = torch.load(os.path.join(inr_save_dir, inr_save_name+'.pt'))
/pscratch/sd/g/gzhao2

In [14]:
z_mean = ode_results['z_mean']
z_std = ode_results['z_std']
z_transform = lambda tensor: (tensor - z_mean) / z_std
z_invtransform = lambda tensor: (tensor * z_std) + z_mean

In [6]:
model_state_dict = ode_results['ode']
model = Derivative(output_dim, latent_dim, hidden, depth).cuda()
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [24]:
testset = GraphNavierStokes(split='test', datanum=20, missing_rate=0.0, latent_dim=latent_dim)
load_graph_modulations(
    testset,
    inr,
    inner_steps=inner_steps,
    alpha=alpha,
    batch_size=2,
)

(-0.0003125140501651913, 0.008481579832732677)

In [27]:
test_loader = DataLoader(
    testset,
    batch_size=20,
    shuffle=True,
    collate_fn=collate_graph_navier_stokes,
    num_workers=1,
)

In [28]:
graph = next(iter(test_loader))
outputs = graph_ode_inr_predict(
    
    model, 
    inr, 
    graph, 
    timestamps=None,
    z_transform=z_transform,
    z_invtransform=z_invtransform
    )
ode_pred = outputs['ode_pred']
inr_pred = outputs['inr_pred']

In [29]:
outputs

{'ode_pred': tensor([[-0.2237],
         [-0.1840],
         [-0.1563],
         ...,
         [ 0.4024],
         [ 0.6231],
         [ 0.8365]]),
 'ode_pred_loss': tensor(0.0147, device='cuda:0', grad_fn=<MeanBackward0>),
 'inr_pred': tensor([[-0.2237],
         [-0.1840],
         [-0.1563],
         ...,
         [ 0.3285],
         [ 0.5535],
         [ 0.7660]]),
 'inr_pred_loss': tensor(0.0044, device='cuda:0', grad_fn=<MeanBackward0>),
 'z_pred_loss': tensor(0.0314, device='cuda:0', grad_fn=<MeanBackward0>)}

In [30]:
import matplotlib.pyplot as plt
import numpy as np

# Assume coordinates and values are given as tensors
# coordinates: (64**2, 2), values: (64**2, 1)
dimage = testset.height

def graph2image(graph, image_pred):
    T = graph.T.sum()
    video_gt = np.zeros((T, dimage, dimage))
    video_pred = np.zeros((T, dimage, dimage))
    
    
    for t in range(T):
        tidx = graph.time ==t
        coordinates = graph.cor[tidx]
        values_gt = graph.feat[tidx]
        values_pred = image_pred[tidx]
        
        coordinates_np = coordinates.numpy()
        values_gt = values_gt.numpy().reshape(-1)
        values_pred = values_pred.numpy().reshape(-1)
        
        for i, (x, y) in enumerate(coordinates_np):
            video_gt[t, x, y] = values_gt[i]
            video_pred[t, x, y] = values_pred[i]
    
    return video_gt, video_pred

def drawimage(graph, value_pred, t, test=False):

    tidx = graph.time ==t
    coordinates = graph.cor[tidx]
    if not test:
        values = graph.feat[tidx]
    else:
        values = value_pred[tidx]

    # Convert tensors to numpy arrays
    coordinates_np = coordinates.numpy()
    values_np = values.numpy().reshape(-1)

    # Create an empty 2D array to hold the pixel values
    image = np.zeros((dimage, dimage))

    # Populate the image using the coordinates and values
    for i, (x, y) in enumerate(coordinates_np):
        image[x, y] = values_np[i]

    # Plot the image using matplotlib
    plt.imshow(image,)  # You can change 'gray' to another colormap if desired
    plt.colorbar()  # Optional: Adds a colorbar to show value scale
    plt.title("64x64 Image from Coordinates and Values")
    plt.show()

In [31]:
video_gt, video_pred = graph2image(graph, ode_pred)
_, video_inr = graph2image(graph, inr_pred)

In [33]:
video_gt.shape

(1000, 64, 64)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
from IPython.display import HTML

# Assuming your tensor is a 3D numpy array of shape (50, 64, 64)
tensor1 = video_gt  # Replace this with your actual tensor
tensor2 = video_inr
tensor3 = video_pred

# Create a figure and axis to display the images
fig, (ax1, ax2, ax3) = plt.subplots(1, 3)

# Display the first frame
im1 = ax1.imshow(tensor1[0], vmax = 1.5, vmin = -1.5)
im2 = ax2.imshow(tensor2[0], vmax = 1.5, vmin = -1.5)
im3 = ax3.imshow(tensor3[0], vmax = 1.5, vmin = -1.5)

ax1.set_title('Ground Truth')
ax2.set_title('INR prediction')
ax3.set_title('ODE prediction')

# Update function for animation
def update(frame):
    im1.set_array(tensor1[frame])
    im2.set_array(tensor2[frame])
    im3.set_array(tensor2[frame])
    return [im1, im2, im3]

# Create an animation
ani = FuncAnimation(fig, update, frames=range(tensor1.shape[0]), blit=True, interval=100)

# Use HTML to display the animation in the notebook
HTML(ani.to_jshtml())


